In [0]:
%pip install datacompy

In [0]:
dbutils.library.restartPython()

In [0]:
import datacompy
import pandas as pd

In [0]:
# Configuration
from pyspark.sql import functions as F, Window
spark.conf.set("spark.sql.session.timeZone", "Europe/Warsaw")
dbutils.widgets.combobox("catalog", "workspace", ["workspace", "dbr_dev"], "Unity Catalog")
dbutils.widgets.combobox("gold_schema", "gold", ["gold", "gabrielajaniszews786_gold"], "Gold schema")

CATALOG       = dbutils.widgets.get("catalog")
GOLD_SCHEMA = dbutils.widgets.get("gold_schema")

consumption_hourly = f"{CATALOG}.{GOLD_SCHEMA}.consumption_hourly"
dim_date = f"{CATALOG}.{GOLD_SCHEMA}.dim_date"
dim_datacenter = f"{CATALOG}.{GOLD_SCHEMA}.dim_datacenter"




In [0]:
# Cloning the tables

for name in [consumption_hourly, dim_datacenter, dim_date]:
    spark.sql(f"CREATE OR REPLACE TABLE {name}_clone AS SELECT * FROM {name}")

In [0]:
spark.sql(f"""
    UPDATE {consumption_hourly}_clone
    SET avg_consumption_kwh = avg_consumption_kwh + 100
    WHERE hour = 12
""")

In [0]:
source_pd = spark.table(consumption_hourly).toPandas()
target_pd = spark.table("consumption_hourly_clone").toPandas()


In [0]:
compare = datacompy.PandasCompare(
    source_pd,
    target_pd,
    join_columns=["site_id", "bidding_zone", "date", "hour"],
    df1_name="source",
    df2_name="target",
)

print(compare.report())

## Mass comparison

In [0]:


# config: table > key
tables_config = {
    consumption_hourly: ["site_id", "bidding_zone", "date", "hour"],
    dim_datacenter:     ["site_id"],
    dim_date:           ["date"],
}

results = []
for src_name, keys in tables_config.items():
    tgt_name = f"{src_name}_clone"
    src = spark.table(src_name).toPandas()
    tgt = spark.table(tgt_name).toPandas()

    cmp = datacompy.PandasCompare(src, tgt, join_columns=keys,
                                  df1_name="source", df2_name="target")
    results.append({
        "table":            src_name.split(".")[-1],
        "rows_source":      src.shape[0],
        "rows_target":      tgt.shape[0],
        "rows_only_source": cmp.df1_unq_rows.shape[0],
        "rows_only_target": cmp.df2_unq_rows.shape[0],
        "rows_matching":    cmp.count_matching_rows(),
        "all_columns_match": cmp.all_columns_match(),
        "all_match":        cmp.matches(),
    })

summary = pd.DataFrame(results)
display(summary)

#spark.createDataFrame(summary).write.mode("overwrite").saveAsTable("recon_summary")

## Pure Spark Reconcilliation

In [0]:
from pyspark.sql import functions as F

In [0]:
# Hashing the rows
def add_row_hash(df, cols):
    norm = [F.coalesce(F.col(c).cast("string"), f.lit("∅")) for c in cols]
    return df.withColumn("row_hash", F.md5(F.concat_ws("|", *norm)))

In [0]:
def reconcile_table(src_name, tgt_name, key_cols):
    src = spark.table(src_name)
    tgt = spark.table(tgt_name)

    cols = src.columns # same order of columns on each side
    tgt = tgt.select(*cols) # unifying the columns order

    src_h = add_row_hash(src, cols).select(*key_cols, F.col("row_hash").alias("src_hash"))
    tgt_h = add_row_hash(tgt, cols).select(*key_cols, F.col("row_hash").alias("tgt_hash"))

    joined = src_h.join(tgt_h, key_cols, "fullouter")
    
    classified = joined.withColumn(
        "status",
        F.when(F.col("src_hash").isNull(), F.lit("only_in_target"))
        .when(F.col("tgt_hash").isNull(), F.lit("only_in_source"))
        .when(F.col("src_hash") == F.col("tgt_hash"), F.lit("match"))
        .otherwise(F.lit("unchanged"))
    )
    counts = {r["status"]: r["count"] for r in classified.groupBy("status").count().collect()}
    return {
        "table":          src_name.split(".")[-1],
        "match":          counts.get("match", 0),
        "changed":        counts.get("changed", 0),
        "only_in_source": counts.get("only_in_source", 0),
        "only_in_target": counts.get("only_in_target", 0),
    }

In [0]:
from pyspark.sql import functions as f

results = [
    reconcile_table(name, f"{name}_clone", keys)
    for name, keys in tables_config.items()
]
summary = spark.createDataFrame(results)
display(summary)